# v0 — `fb_posts` → structured listings (read-only, 10 rows)

This notebook is a **visual checkpoint** before the full v1 Supabase pipeline. It:

1. reads **10 real `fb_posts` rows** from Supabase (READ-ONLY, no writes anywhere),
2. runs the shared `extract.py` engine in **ONE batched LLM call**,
3. shows the transformation step by step: raw text → raw model JSON → normalized
   40-field table → goal-fields side-by-side → summary.

It calls the *exact same engine* the v1 pipeline uses, so what you see here is what v1
will write to `listings_parsed` at scale. **Nothing is written to the database.**


## Setup
Locate the project root (so `import extract` works), load `.env`, and check keys.

In [ ]:
import sys, json
from pathlib import Path

# Find the project root (this notebook lives in notebooks/; the Python modules live
# in src/, and .env is at the project root).
cwd = Path.cwd()
ROOT = cwd if (cwd / "src" / "extract.py").exists() else cwd.parent
assert (ROOT / "src" / "extract.py").exists(), f"could not find src/extract.py from {cwd}"
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import os
import pandas as pd
import extract

missing = [k for k in ("ANTHROPIC_API_KEY", "SUPABASE_URL", "SUPABASE_ANON_KEY")
           if not os.environ.get(k)]
assert not missing, f"Missing in .env: {missing}. Add them and re-run."
print("Project root:", ROOT)
print("extract.py:", len(extract.MODEL_FIELDS), "fields, parser", extract.PARSER_VERSION)
print("env OK")

## Step 1 — read 10 raw `fb_posts` rows (READ-ONLY)
A single `.limit(10)` query. No writes.

In [ ]:
from supabase import create_client

client = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_ANON_KEY"])
res = client.table("fb_posts").select("id,source,text,created_at").limit(10).execute()
raw_rows = res.data or []
print(f"fetched {len(raw_rows)} rows")

# Guard: RLS without an anon SELECT policy (or an empty table) returns [] silently,
# which would later blow up as a confusing KeyError. Fail loudly here instead.
assert raw_rows, ("0 rows returned. Either fb_posts is empty, or the anon key lacks a "
                  "SELECT policy on fb_posts (run STEP 2 of "
                  "chrome_extension/KP-Rentals-Exporter/supabase_schema.sql).")

raw_df = pd.DataFrame(raw_rows)
pd.set_option("display.max_colwidth", 200)
raw_df[["id", "source", "created_at", "text"]]


## Step 2 — run the extraction (ONE LLM call)

`call_claude_batch(texts)` sends all 10 posts in a single request and returns a JSON
array (one object per post). This is the happy path of `extract.extract_batch()`, which
in production also (a) skips texts shorter than 15 chars with **no** call and (b) falls
back to per-row calls if the batch fails — neither should trigger on 10 normal rows.


In [ ]:
texts = [r.get("text") or "" for r in raw_rows]

raw_objects = extract.call_claude_batch(texts)   # <-- the single LLM call
print(f"model returned {len(raw_objects)} objects for {len(texts)} posts")

if len(raw_objects) != len(texts):
    print("! length mismatch -> using robust extract_batch() (adds per-row fallback)")
    normalized = extract.extract_batch(texts)
    raw_objects = None
else:
    normalized = [extract._coerce(o) for o in raw_objects]
print("normalized rows:", len(normalized))


## Step 3 — what the model returned (raw JSON, first 2 posts)
Before any normalization/coercion.

In [ ]:
if raw_objects is not None:
    print(json.dumps(raw_objects[:2], indent=2, ensure_ascii=False))
else:
    print("(batch fell back to per-row; raw array not available)")


## Step 4 — normalized 40-field table

After `_coerce`: enums lowercased (invalid → fallback), list fields joined, `null` kept
as unknown. These columns are exactly what v1 will upsert into `listings_parsed`.


In [ ]:
assert normalized, "no normalized rows"
parsed_df = pd.DataFrame(normalized)[extract.MODEL_FIELDS]
parsed_df


## Step 5 — side-by-side: raw text → GOAL fields

The human sanity-check, focused on what this project is *for*: long-term suitability
(`season`, `year_round`, `min_stay_months`) and sublet potential (`subletting_allowed`),
alongside price/area/intent. Also the best place to spot **batch misalignment** — if a
row's fields don't match its text, the model returned objects out of order.


In [ ]:
key_cols = ["is_offer", "discard_reason", "price_thb", "season", "year_round",
            "min_stay_months", "subletting_allowed", "bedrooms", "area_canonical",
            "post_language"]
side = parsed_df[key_cols].copy()
side.insert(0, "text", [(r.get("text") or "")[:120] for r in raw_rows])
side.insert(0, "id", [r.get("id") for r in raw_rows])
pd.set_option("display.max_colwidth", 130)
side


## Step 6 — summary

How the 10 rows classify, the goal-relevant distributions, and how many survive the
offers filter (`discard_reason IS NULL`) that v1 will use for your actual house search.


In [ ]:
def counts(col, label):
    print(f"{label}:")
    print(parsed_df[col].fillna("(null/unknown)").value_counts(), "\n")

counts("discard_reason", "discard_reason")
counts("season", "season")
counts("year_round", "year_round (full year incl. high season)")
counts("subletting_allowed", "subletting_allowed")
counts("area_canonical", "area_canonical")
counts("property_type", "property_type")

kept = parsed_df["discard_reason"].isna().sum()
print(f"kept as offers (discard_reason is null): {kept} / {len(parsed_df)}")


## Verdict

If the goal fields above look right (prices/beds/areas sensible on real offers;
`season`/`year_round`/`subletting_allowed` reflect the text; obvious non-listings /
wanted / for-sale rows flagged with the right `discard_reason`), the engine is good and
we proceed to **v1** — the full incremental pipeline that writes `listings_parsed` in
Supabase. If something looks off, we tune `extract.py`'s prompt and re-run this notebook
(still just 1 LLM call) before touching the database.
